# Solutions to notebook 4: Linear algebra with symbols

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/sympy-for-engineers/blob/main/solutions/04_linear_algebra.ipynb)

Worked solutions to the exercises at the end of [notebook 4](../notebooks/04_linear_algebra.ipynb).
Try each exercise yourself before reading its solution: the struggle is where the learning happens.

In [1]:
import sympy as sp

sp.init_printing(use_latex="mathjax")

## Exercise 1

Balance $\mathrm{C_2H_5OH + O_2 \rightarrow CO_2 + H_2O}$ and
$\mathrm{KMnO_4 + HCl \rightarrow KCl + MnCl_2 + H_2O + Cl_2}$ with the null-space method.

One row per element, one column per species, products entered as negative. Ethanol
$\mathrm{C_2H_5OH}$ contains 2 C, 6 H and 1 O.

In [2]:
#                     C2H5OH  O2  CO2  H2O
A_ethanol = sp.Matrix([[2,    0,  -1,   0],     # carbon
                       [6,    0,   0,  -2],     # hydrogen
                       [1,    2,  -2,  -1]])    # oxygen

null_vector = A_ethanol.nullspace()[0]
ethanol = null_vector * sp.ilcm(*[term.q for term in null_vector])      # smallest whole numbers
ethanol.T

[1  3  2  3]

In [3]:
#                       KMnO4  HCl  KCl  MnCl2  H2O  Cl2
A_permanganate = sp.Matrix([[1,  0,  -1,   0,    0,   0],     # potassium
                            [1,  0,   0,  -1,    0,   0],     # manganese
                            [4,  0,   0,   0,   -1,   0],     # oxygen
                            [0,  1,   0,   0,   -2,   0],     # hydrogen
                            [0,  1,  -1,  -2,    0,  -2]])    # chlorine

null_vector = A_permanganate.nullspace()[0]
permanganate = null_vector * sp.ilcm(*[term.q for term in null_vector])
permanganate.T

[2  16  2  2  8  5]

In [4]:
# verify: each null space has exactly one vector (a single reaction), every element balances,
# and the coefficients match the ones found by hand
assert len(A_ethanol.nullspace()) == 1 and len(A_permanganate.nullspace()) == 1
assert A_ethanol * ethanol == sp.zeros(3, 1)
assert A_permanganate * permanganate == sp.zeros(5, 1)
assert list(ethanol) == [1, 3, 2, 3]
assert list(permanganate) == [2, 16, 2, 2, 8, 5]

# a count that does not use the matrices: chlorine atoms on each side of the second reaction
assert 16 * 1 == 2 * 1 + 2 * 2 + 5 * 2

**Answer.**
$\mathrm{C_2H_5OH + 3\,O_2 \rightarrow 2\,CO_2 + 3\,H_2O}$ and
$\mathrm{2\,KMnO_4 + 16\,HCl \rightarrow 2\,KCl + 2\,MnCl_2 + 8\,H_2O + 5\,Cl_2}$.

## Exercise 2

In 4.4, make the middle spring $k_c$ different from the outer ones. How do the two
frequencies depend on $k_c$? What happens as $k_c \to 0$?

In [5]:
m, k, kc = sp.symbols("m k k_c", positive=True)

M_mass = sp.diag(m, m)
K_stiff = sp.Matrix([[k + kc, -kc], [-kc, k + kc]])      # each mass feels its wall spring and k_c

modes = (M_mass.inv() * K_stiff).eigenvects()     # (eigenvalue, multiplicity, [vectors])
for omega_sq, _, vectors in modes:
    display(sp.sqrt(omega_sq), vectors[0].T)

√k
──
√m

[1  1]

  ___________
╲╱ k + 2⋅k_c 
─────────────
     √m      

[-1  1]

In [6]:
omega_low, omega_high = [sp.sqrt(omega_sq) for omega_sq, _, _ in modes]

# verify 1: the characteristic equation det(K - omega^2 M) = 0 gives the same frequencies
w = sp.symbols("omega", positive=True)
frequencies = sp.solve((K_stiff - w**2 * M_mass).det(), w)
assert set(frequencies) == {omega_low, omega_high}

# verify 2: k_c = k recovers the lesson's sqrt(k/m) and sqrt(3k/m)
assert omega_low == sp.sqrt(k / m) and omega_high.subs(kc, k) == sp.sqrt(3 * k / m)

# the limits of a very weak and a very stiff middle spring
assert sp.limit(omega_high, kc, 0) == sp.sqrt(k / m)
assert sp.limit(omega_high, kc, sp.oo) == sp.oo
K_stiff.subs(kc, 0), (M_mass.inv() * K_stiff.subs(kc, 0)).eigenvals()

⎛⎡k  0⎤  ⎧k   ⎫⎞
⎜⎢    ⎥, ⎨─: 2⎬⎟
⎝⎣0  k⎦  ⎩m   ⎭⎠

**Answer.** $\omega_1 = \sqrt{k/m}$ with mode shape $(1, 1)$ and
$\omega_2 = \sqrt{(k + 2k_c)/m}$ with mode shape $(-1, 1)$.

The in-phase frequency does not depend on $k_c$ at all, because the middle spring is never
stretched in that mode. The out-of-phase frequency grows with $k_c$. As $k_c \to 0$ it falls to
$\sqrt{k/m}$ as well: the stiffness matrix becomes diagonal, the eigenvalue $k/m$ is repeated,
and the two masses are independent oscillators, each on its own wall spring. A small $k_c$ gives
two nearly equal frequencies, which is the condition for beats.

## Exercise 3

Show that the maximum shear stress in plane stress is half the difference of the principal
stresses.

Rotate the axes by an angle $\theta$. The stress tensor in the rotated axes is
$\mathbf{Q}^T \boldsymbol{\sigma}\,\mathbf{Q}$, where $\mathbf{Q}$ is the rotation matrix of 4.6,
and its off-diagonal entry is the shear stress on the rotated face.

In [7]:
sx, sy, txy, theta = sp.symbols("sigma_x sigma_y tau_xy theta", real=True)
stress = sp.Matrix([[sx, txy], [txy, sy]])

def rot(angle):
    return sp.Matrix([[sp.cos(angle), -sp.sin(angle)], [sp.sin(angle), sp.cos(angle)]])

rotated = sp.simplify(rot(theta).T * stress * rot(theta))
sigma_rot, tau_rot = rotated[0, 0], rotated[0, 1]
tau_rot

  σₓ⋅sin(2⋅θ)   σ_y⋅sin(2⋅θ)                
- ─────────── + ──────────── + τ_xy⋅cos(2⋅θ)
       2             2                      

In [8]:
# Mohr's circle: for every angle the point (sigma, tau) lies on a circle around the centre
radius = sp.sqrt(((sx - sy) / 2) ** 2 + txy**2)
centre = (sx + sy) / 2
assert sp.simplify((sigma_rot - centre) ** 2 + tau_rot**2 - radius**2) == 0

# the principal stresses are centre ± radius, so half their difference is the radius
s1, s2 = centre + radius, centre - radius
assert all(sp.simplify((p - s1) * (p - s2)) == 0 for p in stress.eigenvals())
assert sp.simplify((s1 - s2) / 2 - radius) == 0

So $\tau'^2 = r^2 - (\sigma' - c)^2 \le r^2$: the shear stress can never exceed the radius
$r = (\sigma_1 - \sigma_2)/2$, and reaches it where the normal stress equals the centre value $c$.
To find where that is, start from the principal axes, in which the tensor is diagonal.

In [9]:
sig1, sig2 = sp.symbols("sigma_1 sigma_2", real=True)
tau_principal = sp.simplify((rot(theta).T * sp.diag(sig1, sig2) * rot(theta))[0, 1])
tau_principal

(-σ₁ + σ₂)⋅sin(2⋅θ)
───────────────────
         2         

In [10]:
# verify: the hand formula -(sigma_1 - sigma_2)/2 * sin(2 theta), largest in size at 45 degrees
assert sp.simplify(tau_principal + (sig1 - sig2) / 2 * sp.sin(2 * theta)) == 0
assert sp.simplify(tau_principal.subs(theta, sp.pi / 4) + (sig1 - sig2) / 2) == 0

# verify with the numbers of 4.5 (MPa): scan all orientations in steps of 0.1 degree
state = {sx: 80, sy: -40, txy: 30}
tau_fun = sp.lambdify(theta, tau_rot.subs(state))
tau_max_scan = max(abs(tau_fun(float(sp.rad(tenth)) / 10)) for tenth in range(1800))
tau_max = float(((s1 - s2) / 2).subs(state))
assert abs(tau_max_scan / tau_max - 1) < 1e-5
print(f"principal stresses {float(s1.subs(state)):.2f} and {float(s2.subs(state)):.2f} MPa, "
      f"maximum shear {tau_max:.2f} MPa (scan over angles: {tau_max_scan:.2f} MPa)")

principal stresses 87.08 and -47.08 MPa, maximum shear 67.08 MPa (scan over angles: 67.08 MPa)


**Answer.** In axes rotated by $\theta$ the stresses satisfy
$(\sigma' - c)^2 + \tau'^2 = r^2$ with $c = \tfrac12(\sigma_x + \sigma_y)$ and
$r = \sqrt{\left(\tfrac{\sigma_x - \sigma_y}{2}\right)^2 + \tau_{xy}^2}$. The principal stresses are
$c \pm r$, so

$$\tau_{max} = r = \frac{\sigma_1 - \sigma_2}{2} .$$

Measured from the principal axes, $\tau' = -\tfrac12(\sigma_1 - \sigma_2)\sin 2\theta$, so the
maximum occurs on planes at 45° to the principal directions. For the state of 4.5 it is 67.08 MPa.

## Exercise 4

Write the 3D rotation matrices about $x$ and $z$, and show that they do not commute.

In [11]:
alpha, gamma = sp.symbols("alpha gamma", real=True)

def rot_x(angle):
    return sp.Matrix([[1, 0, 0],
                      [0, sp.cos(angle), -sp.sin(angle)],
                      [0, sp.sin(angle), sp.cos(angle)]])

def rot_z(angle):
    return sp.Matrix([[sp.cos(angle), -sp.sin(angle), 0],
                      [sp.sin(angle), sp.cos(angle), 0],
                      [0, 0, 1]])

rot_x(alpha), rot_z(gamma)

⎛⎡1    0        0   ⎤  ⎡cos(γ)  -sin(γ)  0⎤⎞
⎜⎢                  ⎥  ⎢                  ⎥⎟
⎜⎢0  cos(α)  -sin(α)⎥, ⎢sin(γ)  cos(γ)   0⎥⎟
⎜⎢                  ⎥  ⎢                  ⎥⎟
⎝⎣0  sin(α)  cos(α) ⎦  ⎣  0        0     1⎦⎠

In [12]:
# the commutator: zero only if the order of the two rotations does not matter
commutator = sp.simplify(rot_x(alpha) * rot_z(gamma) - rot_z(gamma) * rot_x(alpha))
assert commutator != sp.zeros(3)
commutator

⎡         0           (cos(α) - 1)⋅sin(γ)    -sin(α)⋅sin(γ)   ⎤
⎢                                                             ⎥
⎢(cos(α) - 1)⋅sin(γ)           0           (cos(γ) - 1)⋅sin(α)⎥
⎢                                                             ⎥
⎣   sin(α)⋅sin(γ)     (cos(γ) - 1)⋅sin(α)           0         ⎦

In [13]:
# verify 1: both are proper rotations (inverse = transpose, determinant 1)
for Q in (rot_x(alpha), rot_z(gamma)):
    assert sp.simplify(Q.T * Q) == sp.eye(3) and sp.simplify(Q.det()) == 1

# verify 2: a case that can be done by hand. Turn the x unit vector by 90 degrees about each axis.
# z first: e_x -> e_y, then x: e_y -> e_z.    x first: e_x stays, then z: e_x -> e_y.
ex = sp.Matrix([1, 0, 0])
quarter = sp.pi / 2
assert rot_x(quarter) * rot_z(quarter) * ex == sp.Matrix([0, 0, 1])
assert rot_z(quarter) * rot_x(quarter) * ex == sp.Matrix([0, 1, 0])

# verify 3: the commutator vanishes when either angle is zero
assert commutator.subs(alpha, 0) == sp.zeros(3) and commutator.subs(gamma, 0) == sp.zeros(3)

**Answer.**

$$\mathbf{R}_x(\alpha) = \begin{pmatrix} 1 & 0 & 0 \\ 0 & \cos\alpha & -\sin\alpha \\ 0 & \sin\alpha & \cos\alpha \end{pmatrix},
\qquad
\mathbf{R}_z(\gamma) = \begin{pmatrix} \cos\gamma & -\sin\gamma & 0 \\ \sin\gamma & \cos\gamma & 0 \\ 0 & 0 & 1 \end{pmatrix}$$

The difference $\mathbf{R}_x\mathbf{R}_z - \mathbf{R}_z\mathbf{R}_x$ is not the zero matrix, so the
order matters. With two quarter turns, the $x$ unit vector ends up along $z$ in one order and along
$y$ in the other. Rotations in a plane (4.6) commute; rotations about different axes in space do
not, except when one of the angles is zero.

## Exercise 5

Repeat 4.7 for $\ddot x + 2\dot x + x = 0$, whose matrix is `[[0, 1], [-1, -2]]`. The
eigenvalue is repeated (critical damping). How does the form of the solution change?

In [14]:
t = sp.symbols("t", nonnegative=True)
A_sys = sp.Matrix([[0, 1], [-1, -2]])

A_sys.eigenvects()        # one eigenvalue of multiplicity 2, with only one eigenvector

⎡⎛       ⎡⎡-1⎤⎤⎞⎤
⎢⎜-1, 2, ⎢⎢  ⎥⎥⎟⎥
⎣⎝       ⎣⎣1 ⎦⎦⎠⎦

In [15]:
x0 = sp.Matrix([1, 0])                       # released from rest at x = 1
x_t = sp.simplify(sp.exp(A_sys * t) * x0)
x_t

⎡         -t⎤
⎢(t + 1)⋅ℯ  ⎥
⎢           ⎥
⎢      -t   ⎥
⎣  -t⋅ℯ     ⎦

In [16]:
# verify: the matrix equation and the initial condition, as in 4.7
assert sp.simplify(sp.diff(x_t, t) - A_sys * x_t) == sp.zeros(2, 1)
assert x_t.subs(t, 0) == x0

# verify: the position satisfies the original second-order equation, the second component is
# its derivative, and the result equals the textbook critically damped response (1 + t) exp(-t)
position = x_t[0]
assert sp.simplify(sp.diff(position, t, 2) + 2 * sp.diff(position, t) + position) == 0
assert sp.simplify(sp.diff(position, t) - x_t[1]) == 0
assert sp.simplify(position - (1 + t) * sp.exp(-t)) == 0

# the repeated eigenvalue has a single eigenvector, so the matrix cannot be diagonalized
assert A_sys.eigenvals() == {-1: 2} and not A_sys.is_diagonalizable()
[float(position.subs(t, time)) for time in (0, 1, 2, 5)]

[1.0, 0.7357588823428847, 0.40600584970983805, 0.040427681994512805]

**Answer.** $x(t) = (1 + t)\,e^{-t}$ and $\dot x(t) = -t\,e^{-t}$.

In 4.7 the two distinct eigenvalues gave a sum of two exponentials, $e^{-t}$ and $e^{-2t}$. Here
the eigenvalue $-1$ is repeated and has only one eigenvector, so a second exponential is not
available. Its place is taken by $t\,e^{-t}$: an exponential multiplied by a polynomial in $t$.
The response still decays without oscillating, and the velocity never changes sign, so the mass
returns to rest without overshoot.